# The dark plasmidome

`cryptic_plasmids.ipynb` (cells 16–22) established that **71.5% of the coding capacity of the
payload-free small plasmidome is unannotatable**, and that **43.5% of those plasmids are dark end
to end**. That result is compatible with two opposite readings, and it cannot separate them:

- **idiosyncratic junk** — nearly every dark ORF unique, consistent with non-adaptive persistence
- **recurrent dark families** — a bounded set of uncharacterized protein families appearing over and
  over, i.e. an annotation gap rather than an absence of function

This notebook separates them by clustering the dark proteome itself. It reads the outputs of
`scripts/extract_plasann_proteins.py` → `scripts/cluster_dark_orfs.sh`, documented in
`reports/dark_orf_clustering.md`; nothing here re-runs MMseqs2.

**Answer, up front: the dark plasmidome is recurrent, not idiosyncratic** — and the recurrence is
not clonal redundancy.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "data" / "dark_orf_run").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
D = ROOT / "data" / "dark_orf_run"
PP = ROOT / "data" / "plasmidscope_primary"

# Palette and axis helper carried over from cryptic_plasmids.ipynb so the two notebooks match.
S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dcdbd4"

def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)

# Lineage (mob_cluster) and habitat labels for the 71,414 payload-free small plasmids.
meta = (pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False,
                    usecols=["plasmid_id", "mob_cluster", "hab_sub", "size_bp"])
        .set_index("plasmid_id"))

# dark30 = dark ORF proteins clustered alone;  mix30 = dark + named clustered together.
# Both from scripts/cluster_dark_orfs.sh (MMseqs2, 30% identity / 80% bidirectional coverage).
dark = pd.read_csv(D / "dark30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
dark["plasmid"] = dark["member"].str.rsplit("|", n=1).str[0]
dark = dark.join(meta, on="plasmid")

mix = pd.read_csv(D / "mix30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
mix["kind"] = mix["member"].str.rsplit("|", n=1).str[1]

N_PLASMID, N_DARK = 71_414, len(dark)
N_LINEAGE = meta.loc[dark["plasmid"].unique(), "mob_cluster"].nunique()

print(f"payload-free small plasmids   {N_PLASMID:,}")
print(f"dark ORF proteins             {N_DARK:,}  on {dark['plasmid'].nunique():,} plasmids")
print(f"dark families (30% id/80% cov){dark['rep'].nunique():>8,}")
print(f"MOB lineages in the subset    {N_LINEAGE:,}")
print(f"\nall CDS proteins co-clustered {len(mix):,} "
      f"({(mix['kind'] == 'D').sum():,} dark + {(mix['kind'] == 'N').sum():,} named)")
print(f"co-clustering families        {mix['rep'].nunique():,}")


## 1. The dark proteome is recurrent, not idiosyncratic

If the dark plasmidome were junk accumulating independently on each replicon, almost every ORF would
be its own family. It isn't.


In [ ]:
# R1 — is the dark proteome recurrent, or is every ORF unique?
size = dark.groupby("rep").size()

TIERS = [(1, 1, "singleton"), (2, 4, "2–4"), (5, 9, "5–9"), (10, 49, "10–49"),
         (50, 99, "50–99"), (100, 999, "100–999"), (1000, 10**9, "≥1000")]
rows = []
for lo, hi, lab in TIERS:
    sel = size.between(lo, hi)
    rows.append({"tier": lab, "families": int(sel.sum()), "dark ORFs": int(size[sel].sum())})
tier = pd.DataFrame(rows).set_index("tier")
tier["% of dark ORFs"] = tier["dark ORFs"] / N_DARK * 100

print(f"{N_DARK:,} dark ORF proteins -> {size.size:,} families\n")
print(tier.round(1).to_string())

rec = size[size >= 2].sum()
print(f"\nrecurrent (family >=2 members): {rec:,}  ({rec / N_DARK * 100:.1f}% of dark ORFs)")
print(f"unique  (singleton family):     {N_DARK - rec:,}  ({(N_DARK - rec) / N_DARK * 100:.1f}%)")

# Concentration: how few families hold most of the dark coding capacity.
srt = size.sort_values(ascending=False)
cum = srt.cumsum() / N_DARK
for f in (0.25, 0.50, 0.75):
    print(f"  {f * 100:.0f}% of dark ORFs sit in the top {int((cum < f).sum()) + 1:,} families")

# Protein length — are these real proteins or spurious short calls?
lens = np.array([len(l.strip()) for l in open(D / "dark_orfs.faa") if not l.startswith(">")])
print(f"\nlength (aa): q25 {np.percentile(lens, 25):.0f} | median {np.median(lens):.0f} | "
      f"q75 {np.percentile(lens, 75):.0f} | max {lens.max():,}")
print(f"  under 50 aa: {(lens < 50).mean() * 100:.1f}%   under 100 aa: {(lens < 100).mean() * 100:.1f}%")


In [ ]:
# Figure R1. Grey marks the non-recurrent (singleton) fraction, as absence does elsewhere.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.1]})

x = np.arange(len(tier))
vals = tier["dark ORFs"].values
axA.bar(x, vals, width=0.5,
        color=[NEUT if t == "singleton" else S1 for t in tier.index])
for i, (v, f) in enumerate(zip(vals, tier["families"])):
    axA.text(i, v + vals.max() * 0.02, f"{v / N_DARK * 100:.1f}%\n{f:,} fam",
             ha="center", fontsize=8, color=INK2)
axA.set(xticks=x, xticklabels=tier.index, yticks=[], ylim=(0, vals.max() * 1.28))
axA.set_xlabel("proteins in the family", fontsize=9, color=INK2)
axA.set_title(f"Dark ORFs by family size  (n = {N_DARK:,})\n"
              f"{rec / N_DARK * 100:.1f}% sit in a family with at least one other member",
              fontsize=11, color=INK, loc="left")
tidy(axA)

# B — concentration curve. Log x because family ranks span five orders of magnitude.
rank = np.arange(1, srt.size + 1)
axB.plot(rank, cum.values * 100, color=S1, linewidth=1.8)
axB.fill_between(rank, cum.values * 100, color=S1, alpha=0.10)
n10 = int((size >= 10).sum())
y10 = cum.values[n10 - 1] * 100
axB.plot([n10], [y10], "o", color=INK, markersize=5)
axB.annotate(f"the {n10:,} families with ≥10 members\nhold {y10:.1f}% of the dark proteome",
             xy=(n10, y10), xytext=(n10 * 1.5, y10 - 26), fontsize=9, color=INK,
             arrowprops=dict(arrowstyle="-", color=INK2, linewidth=0.8))
axB.set(xscale="log", xlim=(1, srt.size), ylim=(0, 101), yticks=[0, 25, 50, 75, 100])
axB.set_xlabel("families, ranked by size (log)", fontsize=9, color=INK2)
axB.set_ylabel("cumulative % of dark ORFs", fontsize=9, color=INK2)
axB.grid(axis="y", color=GRID, linewidth=1)
axB.set_title(f"Concentration across {size.size:,} families\n"
              "a long tail, but the mass is in a few thousand", fontsize=11, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


## 2. The recurrence is not clonal redundancy

Recurrence on its own proves nothing — the same lineage sequenced a thousand times would produce it.
This is the trap that caught the Acinetobacter signal in `cryptic_plasmids.ipynb` cell 10, where a
genus-level pattern collapsed to two MOB clusters. The control is whether a family spans
*independent* lineages.


In [ ]:
# R2 — is the recurrence real, or one lineage sequenced many times?
# Same trap as cryptic_plasmids.ipynb cell 10, where an Acinetobacter signal collapsed to two
# MOB clusters. mob_cluster (MOB-suite primary cluster) is the lineage proxy.
fam = dark.groupby("rep").agg(proteins=("member", "size"), plasmids=("plasmid", "nunique"),
                              lineages=("mob_cluster", "nunique"),
                              habitats=("hab_sub", "nunique"))
multi = fam[fam.proteins >= 2]

LT = [(1, 1, "1 lineage"), (2, 4, "2–4"), (5, 9, "5–9"), (10, 49, "10–49"), (50, 10**9, "≥50")]
rows = []
for lo, hi, lab in LT:
    sel = multi.lineages.between(lo, hi)
    rows.append({"lineages": lab, "families": int(sel.sum()),
                 "dark ORFs": int(multi.loc[sel, "proteins"].sum())})
lin = pd.DataFrame(rows).set_index("lineages")
lin["% of dark ORFs"] = lin["dark ORFs"] / N_DARK * 100

print(f"families with >=2 members: {len(multi):,}\n")
print(lin.round(1).to_string())

cross = multi.loc[multi.lineages >= 2, "proteins"].sum()
print(f"\nconfined to one lineage:   {(multi.lineages == 1).sum():,} families "
      f"({(multi.lineages == 1).mean() * 100:.1f}%)")
print(f"spanning >=2 lineages:     {(multi.lineages >= 2).sum():,} families "
      f"({(multi.lineages >= 2).mean() * 100:.1f}%)")
print(f"  -> these hold {cross:,} dark ORFs = {cross / N_DARK * 100:.1f}% of the dark proteome")

big = fam[fam.proteins >= 10]
print(f"\nsubstantial families (>=10 members): {len(big):,}")
print(f"  spanning >=2 lineages: {(big.lineages >= 2).sum():,} ({(big.lineages >= 2).mean() * 100:.1f}%)")
print(f"  median lineages {big.lineages.median():.0f} | median habitats {big.habitats.median():.0f}")

print(f"\nmost widespread families (of {N_LINEAGE:,} lineages, {dark['hab_sub'].nunique()} habitats):")
top = fam.nlargest(10, "lineages").copy()
top["% lineages"] = top["lineages"] / N_LINEAGE * 100
print(top.round(1).to_string())


In [ ]:
# Figure R2. Grey = the single-lineage families, i.e. exactly the ones clonal redundancy explains.
# Bars encode dark ORFs (not family counts) so panel A stays comparable with figure R1.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.15]})

x = np.arange(len(lin))
vals = lin["dark ORFs"].values
axA.bar(x, vals, width=0.5, color=[NEUT if i == 0 else S1 for i in range(len(lin))])
for i, (v, f) in enumerate(zip(vals, lin["families"])):
    axA.text(i, v + vals.max() * 0.02, f"{v / N_DARK * 100:.1f}%\n{f:,} fam",
             ha="center", fontsize=8, color=INK2)
axA.set(xticks=x, xticklabels=lin.index, yticks=[], ylim=(0, vals.max() * 1.26))
axA.set_xlabel("MOB lineages the family spans", fontsize=9, color=INK2)
axA.set_title(f"Dark ORFs by lineage spread of their family  ({len(multi):,} recurrent families)\n"
              f"{cross / N_DARK * 100:.1f}% of the dark proteome crosses lineage boundaries",
              fontsize=11, color=INK, loc="left")
tidy(axA)

# B — the relationship itself: bigger families are not one lineage counted repeatedly.
hb = axB.hexbin(multi.proteins, multi.lineages, xscale="log", yscale="log", gridsize=34,
                bins="log", cmap="Blues", mincnt=1, linewidths=0.2)
axB.set_xlabel("proteins in the family (log)", fontsize=9, color=INK2)
axB.set_ylabel("MOB lineages spanned (log)", fontsize=9, color=INK2)
lim = np.array([2, multi.proteins.max()])
axB.plot(lim, lim, color=INK2, linewidth=1, linestyle=(0, (4, 3)), zorder=3)
cb = fig.colorbar(hb, ax=axB, pad=0.02)
cb.set_label("families", fontsize=8, color=INK2)
cb.ax.tick_params(colors=INK2, labelsize=8, length=0)
cb.outline.set_visible(False)
axB.set_title("Family size vs lineage spread\n"
              "dashed = one lineage per protein, the ceiling; mass sits well below it "
              "but climbs with size", fontsize=11, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


## 3. The dark families have no named homolog in the dataset

A cheaper explanation is still open: perhaps PlasAnn *can* name these proteins and simply failed to
on these particular plasmids. Co-clustering the dark and named proteomes together tests it — a dark
ORF landing in a family with a named member means the answer was already in the annotation.


In [ ]:
# R3 — are the dark families just proteins PlasAnn names elsewhere but missed here?
# mix30 clustered dark and named proteins together. A dark ORF landing in a family with a named
# member means PlasAnn had the answer and its own annotation was inconsistent.
kind = mix.groupby("rep")["kind"].agg(n="size", nD=lambda s: (s == "D").sum(),
                                      nN=lambda s: (s == "N").sum())
has_named = kind[(kind.nD > 0) & (kind.nN > 0)]
dark_only = kind[(kind.nD > 0) & (kind.nN == 0)]

print(f"{len(mix):,} CDS proteins co-clustered -> {len(kind):,} families\n")
part = pd.DataFrame({
    "families": [len(has_named), len(dark_only), int((dark_only.n >= 10).sum())],
    "dark ORFs": [int(has_named.nD.sum()), int(dark_only.nD.sum()),
                  int(dark_only.loc[dark_only.n >= 10, "nD"].sum())],
}, index=["family has a named member", "dark-only family",
          "  ...of those, families >=10 members"])
part["% of dark ORFs"] = part["dark ORFs"] / N_DARK * 100
print(part.round(1).to_string())

print(f"\n{dark_only.nD.sum() / N_DARK * 100:.1f}% of dark ORFs have no named homolog anywhere "
      f"in the annotated set.")
print(f"{(dark_only.n >= 10).sum():,} dark-only families have >=10 members — the candidate list.")

# Lineage spread of the dark-only families, computed on this clustering (not carried over from R2).
mx = mix[mix["kind"].eq("D")].copy()
mx["plasmid"] = mx["member"].str.rsplit("|", n=2).str[0]
mx = mx[mx["rep"].isin(dark_only.index)].join(meta, on="plasmid")
dofam = mx.groupby("rep").agg(proteins=("member", "size"), plasmids=("plasmid", "nunique"),
                              lineages=("mob_cluster", "nunique"),
                              habitats=("hab_sub", "nunique"))
dobig = dofam[dofam.proteins >= 10]
print(f"\nof the {len(dobig):,} dark-only families with >=10 members: "
      f"{(dobig.lineages >= 2).sum():,} ({(dobig.lineages >= 2).mean() * 100:.1f}%) span >=2 lineages, "
      f"median {dobig.lineages.median():.0f} lineages / {dobig.habitats.median():.0f} habitats")
print("\ntop dark-only families by lineage spread:")
print(dofam.nlargest(10, "lineages").to_string())


In [ ]:
# Figure R3. Grey = dark ORFs explained away by an existing name; blue = the genuinely unnamed.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.25]})

# A — one stacked bar partitioning all N_DARK dark ORFs by what the co-clustering explains.
# The total is read from the data, not hardcoded: it was 379,397 before the duplicate-record
# fix of 2026-08-28 (reports/dark_orf_clustering.md "Corrections") and is 378,552 after.
seg = [("in a family with a named member", int(has_named.nD.sum()), NEUT, INK),
       ("dark-only family, <10 members", int(dark_only.loc[dark_only.n < 10, "nD"].sum()), S3, "white"),
       ("dark-only family, ≥10 members", int(dark_only.loc[dark_only.n >= 10, "nD"].sum()), S1, "white")]
left = 0.0
for label, v, colour, lab_ink in seg:
    pct = v / N_DARK * 100
    axA.barh([0], [pct], left=left, height=0.40, color=colour, label=label,
             edgecolor="white", linewidth=1.5)
    axA.text(left + pct / 2, 0, f"{pct:.1f}%\n{v:,}", ha="center", va="center",
             fontsize=9, color=lab_ink)
    left += pct
axA.set(xlim=(0, 100), xticks=[0, 25, 50, 75, 100], yticks=[], ylim=(-0.35, 0.35))
axA.set_xlabel(f"% of the {N_DARK:,} dark ORFs", fontsize=9, color=INK2)
axA.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="upper center",
           bbox_to_anchor=(0.5, -0.18), ncol=1)
axA.set_title("What co-clustering with the named proteome explains\n"
              f"{dark_only.nD.sum() / N_DARK * 100:.1f}% has no named homolog at all",
              fontsize=11, color=INK, loc="left")
tidy(axA)

# B — the deliverable: the widest-ranging families that remain unnamed.
tb = dofam.nlargest(12, "lineages").iloc[::-1]
y = np.arange(len(tb))
axB.barh(y, tb["lineages"].values, height=0.42, color=S1)
for i, (lg, pl) in enumerate(zip(tb["lineages"], tb["plasmids"])):
    axB.text(lg + tb["lineages"].max() * 0.015, i, f"{lg}  ({pl:,} plasmids)",
             va="center", fontsize=8, color=INK2)
axB.set(yticks=y, yticklabels=[r.rsplit("|", 2)[0][:34] for r in tb.index],
        xlim=(0, tb["lineages"].max() * 1.30))
axB.set_xlabel(f"MOB lineages spanned (of {N_LINEAGE:,})", fontsize=9, color=INK2)
axB.set_title(f"Widest-ranging dark-only families\n"
              f"{(dobig.lineages >= 2).sum():,} of {len(dobig):,} substantial dark-only families "
              f"cross lineages", fontsize=11, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


## 4. Pfam-A: is this a PlasAnn artifact, or genuinely uncharacterized?

Sections 1–3 only ever compared PlasAnn against itself, so "dark" could still have meant nothing
more than a gap in one tool. This section searches every dark family representative against
**Pfam-A 38.2** (30,134 families) with `hmmsearch --cut_ga` — Pfam's own curated per-family
gathering thresholds, so no e-value is chosen by us.

Two design points make the result interpretable:

- **A positive control.** The 78,397 proteins PlasAnn *did* name are searched too. Without them a
  low dark hit rate is ambiguous: it could mean the dark families are novel, or merely that plasmid
  proteins in general are poorly covered by Pfam.
- **A propagation check.** Every member of 200 sampled families is searched, to test whether a
  representative's Pfam status can stand for its whole family.

Full methodology, including two bugs found and fixed during this run, is in
`reports/pfam_dark_validation.md`.


In [ ]:
# R4 — Pfam-A 38.2 verdict. The search is scripts/pfam_array.sbatch (hmmsearch --cut_ga,
# 88-task SLURM array over Pfam 38.2 / 30,134 families); tables by scripts/summarize_pfam.py
# and scripts/pfam_verdict.py. Everything below is recomputed here, not pasted.
import re
PF = ROOT / "data" / "pfam_run"
per = pd.read_csv(PF / "pfam_per_seq.tsv", sep="\t")
man = dict(l.split("\t") for l in (PF / "query_manifest.txt").read_text().splitlines())

rate = {s: (per["set"] == s).sum() / int(man[s]) * 100 for s in ("NAMED", "DARKREP")}
print("Pfam-A 38.2 hit rate at the curated gathering threshold (--cut_ga):")
print(f"  NAMED   positive control, {int(man['NAMED']):,} proteins PlasAnn DID name: "
      f"{rate['NAMED']:.1f}%")
print(f"  DARKREP {int(man['DARKREP']):,} dark family representatives:               "
      f"{rate['DARKREP']:.1f}%")
print("  -> the assay detects plasmid proteins fine. The dark set is genuinely depleted.\n")

# Every dark ORF, cross-tabbed by what can name it: a PlasAnn homolog, Pfam, both, neither.
# NOTE: this join only works because extract_plasann_proteins.py indexes every translated CDS,
# so a dark ORF carries the same id in dark_orfs.faa and all_cds.faa. See the report.
darkhit = per[per["set"] == "DARKREP"].copy()
darkhit["rep"] = darkhit["target"].str.split("::").str[1]
hitset = set(darkhit["rep"])
dmap = dict(zip(dark["member"], dark["rep"]))
dark_only = set(mix.groupby("rep")["kind"].apply(lambda s: (s == "N").sum() == 0)
                .pipe(lambda x: x[x].index))
dm = mix[mix["kind"].eq("D")].copy()
dm["dmem"] = dm["member"].str.rsplit("|", n=1).str[0]
assert dm["dmem"].isin(dmap).all(), "id join broken — see reports/pfam_dark_validation.md"
dm["named_homolog"] = ~dm["rep"].isin(dark_only)
dm["pfam"] = dm["dmem"].map(dmap).isin(hitset)

CT = pd.crosstab(dm["named_homolog"], dm["pfam"]) / len(dm) * 100
CT.index = ["dark-only (no named homolog)", "has a PlasAnn-named homolog"]
CT.columns = ["no Pfam match", "Pfam match"]
print("what can name a dark ORF (% of the dark proteome):")
print(CT.round(1).to_string())
neither = CT.loc["dark-only (no named homolog)", "no Pfam match"]
rescued = CT.loc["dark-only (no named homolog)", "Pfam match"]
print(f"\n  {rescued:.1f}% — PlasAnn could not name it but Pfam can: annotation failure")
print(f"  {neither:.1f}% — neither can: the genuine residue")

# Of what Pfam rescued, how much sits in a category PlasAnn already has?
SCOPE_PAT = [("replication", r"^(Rep_|Rep\d|Rep3|RepL|RepA|Replicase|Rep_OBD|Rep_trans|RepB|Prim)"),
             ("mobilization / conjugation",
              r"^(Mob_|MobA|MobC|Relaxase|NikA|TraD|TraM|TrwB|T4SS|VirD|Conjug|MobP|MobQ)"),
             ("toxin-antitoxin",
              r"(toxin|antitoxin|RelB|ParE|ParD|HigA|HigB|YoeB|YafQ|VapB|VapC|MazE|MazF|Gp49)"),
             ("transposition / recombination",
              r"^(Transpos|DDE|Resolvase|Phage_integrase|Integrase|Recombinase|HTH_Tnp|rve|IS\d)"),
             ("partition / maintenance", r"^(ParA|ParB|CbiA|ParM|SopA|SopB|CcdB|StbA|Soj)")]

def scope_of(name):
    for lab, pat in SCOPE_PAT:
        if re.search(pat, name, re.I):
            return lab
    return "outside PlasAnn's categories"

size = dark.groupby("rep").size()
darkhit["members"] = darkhit["rep"].map(size)
darkhit["scope"] = darkhit["pfam_name"].map(scope_of)
SCOPE = (darkhit.groupby("scope")["members"].sum().sort_values(ascending=False)
         / darkhit["members"].sum() * 100)
inside = SCOPE.drop("outside PlasAnn's categories").sum()
print("\nwhat Pfam rescued, by whether PlasAnn has a category for it (% of rescued ORFs):")
print(SCOPE.round(1).to_string())
print(f"\n  {inside:.1f}% of rescues fall inside a category PlasAnn already annotates "
      f"= {inside / 100 * CT.values[:, 1].sum():.1f}% of the whole dark proteome.")


In [ ]:
# Figure R4. Blue = still unexplained after Pfam; yellow = Pfam rescued it; grey = already had
# a named homolog. Panel C: orange = PlasAnn had a category for it and still called it "ORF".
fig = plt.figure(figsize=(13.5, 8.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15], hspace=0.55, wspace=0.22)
axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, :])

bars = [("PlasAnn-named\n(positive control)", rate["NAMED"], NEUT),
        ("dark family\nrepresentatives", rate["DARKREP"], S1)]
x = np.arange(2)
axA.bar(x, [v for _, v, _ in bars], width=0.42, color=[c for _, _, c in bars])
for i, (_, v, _) in enumerate(bars):
    axA.text(i, v + 2, f"{v:.1f}%", ha="center", fontsize=10, color=INK2)
axA.set(xticks=x, xticklabels=[l for l, _, _ in bars], yticks=[], ylim=(0, 100))
axA.set_title("Pfam-A hit rate (--cut_ga)\nthe assay detects plasmid proteins fine — "
              "the dark set is genuinely depleted", fontsize=10, color=INK, loc="left")
tidy(axA)

seg = [(CT.loc["dark-only (no named homolog)", "no Pfam match"], S1, "white", "nothing names it"),
       (CT.loc["dark-only (no named homolog)", "Pfam match"], S4, "white", "Pfam only"),
       (CT.loc["has a PlasAnn-named homolog", "Pfam match"], NEUT, INK, "named + Pfam"),
       (CT.loc["has a PlasAnn-named homolog", "no Pfam match"], "#d4d3cc", INK, "named only")]
left = 0.0
for v, colour, ink, lab in seg:
    axB.barh([0], [v], left=left, height=0.40, color=colour, edgecolor="white", linewidth=1.5,
             label=f"{lab}  {v:.1f}%")
    if v >= 5:
        axB.text(left + v / 2, 0, f"{v:.1f}%", ha="center", va="center", fontsize=9, color=ink)
    left += v
axB.set(xlim=(0, 100), xticks=[0, 25, 50, 75, 100], yticks=[], ylim=(-0.34, 0.34))
axB.set_xlabel("% of the dark proteome", fontsize=9, color=INK2)
axB.legend(frameon=False, fontsize=8, labelcolor=INK2, ncol=2,
           loc="upper center", bbox_to_anchor=(0.5, -0.30))
axB.set_title("What can name a dark ORF?", fontsize=10, color=INK, loc="left")
tidy(axB)

ks = list(SCOPE.index)
y = np.arange(len(ks))[::-1]
axC.barh(y, SCOPE.values, height=0.42,
         color=[NEUT if k.startswith("outside") else S2 for k in ks])
for yi, k in zip(y, ks):
    axC.text(SCOPE[k] + 0.8, yi, f"{SCOPE[k]:.1f}%", va="center", fontsize=9, color=INK2)
axC.set(yticks=y, yticklabels=[k.replace(" / ", " /\n") for k in ks],
        xlim=(0, SCOPE.max() * 1.15))
axC.set_xlabel("% of the dark ORFs Pfam rescued", fontsize=9, color=INK2)
axC.grid(axis="x", color=GRID, linewidth=1)
axC.set_title("What Pfam rescued — orange = a domain inside a category PlasAnn already annotates\n"
              f"{inside:.1f}% of rescues are proteins PlasAnn had a slot for and still called "
              f'"Open reading frame"', fontsize=10, color=INK, loc="left")
tidy(axC)

plt.show()


## 5. What Pfam says those proteins actually are

Section 4 counted *how many* dark families Pfam can name and split them by whether the domain falls
inside a category PlasAnn already has — but that split was **our** regex. This section uses Pfam's
own annotation instead: family description, entry type, and **clan** (Pfam's curated superfamily
grouping), with clan names from `Pfam-A.clans.tsv.gz`.

One thing to watch: **a Pfam hit is not automatically a function.** A match to a Domain of Unknown
Function names the family and still tells you nothing about what the protein does, so DUF hits are
counted separately below rather than folded in with the real annotations.


In [ ]:
# R5 — what ARE the Pfam-labelled families? Uses Pfam's own annotation (family description,
# entry type, and clan), not a classification of ours. Clan names come from Pfam-A.clans.tsv.gz;
# the "inside/outside PlasAnn's categories" split in cell R4 was our regex — this is not.
import gzip
PFREF = ROOT / "data" / "refs" / "pfam"

rec, cur = {}, {}
for line in gzip.open(PFREF / "Pfam-A.hmm.dat.gz", "rt"):
    if line.startswith("//"):
        if cur:
            rec[cur.get("AC", "").split(".")[0]] = cur
        cur = {}
    elif line.startswith("#=GF "):
        tag, val = line[5:7], line[8:].strip()
        if tag in ("ID", "AC", "DE", "TP", "CL"):
            cur[tag] = val
pfmeta = pd.DataFrame(rec).T[["ID", "DE", "TP", "CL"]]

clan_name = {}
with gzip.open(PFREF / "Pfam-A.clans.tsv.gz", "rt") as fh:
    for line in fh:
        p = line.rstrip("\n").split("\t")
        if len(p) >= 3 and p[1]:
            clan_name[p[1]] = p[2]

fn = darkhit.copy()                       # DARKREP rows with a Pfam hit, from cell R4
fn["acc"] = fn["pfam_acc"].str.split(".").str[0]
fn = fn.join(pfmeta, on="acc")
fn["clan"] = fn["CL"].map(clan_name)
RESCUED = fn["members"].sum()
print(f"{len(fn):,} dark families carry a Pfam hit, covering {RESCUED:,} dark ORFs "
      f"({RESCUED / len(dark) * 100:.1f}% of the dark proteome)")
print(f"distinct Pfam families involved: {fn['acc'].nunique():,}\n")

# The catch: a hit to a Domain of Unknown Function names the family but not the function.
UNK = (fn["ID"].str.contains(r"^DUF|^UPF", na=False)
       | fn["DE"].str.contains(r"unknown function|uncharacteri|hypothetical", case=False, na=False))
fn["known_function"] = ~UNK
print("CAVEAT — a Pfam hit is not automatically a function:")
print(f"  hits to DUF / unknown-function families: {UNK.sum():,} dark families "
      f"({fn.loc[UNK, 'members'].sum():,} dark ORFs, "
      f"{fn.loc[UNK, 'members'].sum() / RESCUED * 100:.1f}% of rescued), "
      f"across {fn.loc[UNK, 'acc'].nunique():,} distinct DUFs")
print(f"  hits to characterised families:          {(~UNK).sum():,} dark families "
      f"({fn.loc[~UNK, 'members'].sum():,} dark ORFs, "
      f"{fn.loc[~UNK, 'members'].sum() / RESCUED * 100:.1f}%)\n")

print("Pfam entry type, by dark ORF mass:")
print(fn.groupby("TP")["members"].sum().sort_values(ascending=False).to_string())

TOPF = (fn.groupby(["ID", "DE"])["members"].sum().sort_values(ascending=False).head(20))
print(f"\ntop 20 Pfam families by dark ORF mass ({TOPF.sum() / RESCUED * 100:.0f}% of rescued):")
for (i, d), v in TOPF.items():
    print(f"  {v:>6,}  {i:<18} {d[:60]}")

CLAN = fn.groupby("clan")["members"].sum().sort_values(ascending=False)
noclan = fn.loc[fn["clan"].isna(), "members"].sum()
print("\ntop Pfam clans (Pfam's own superfamily grouping) by dark ORF mass:")
print(CLAN.head(12).to_string())
print(f"  no clan assigned: {noclan:,} dark ORFs ({noclan / RESCUED * 100:.1f}% of rescued)")


In [ ]:
# Figure R5. Grey has ONE meaning here, the same as elsewhere in the notebook: absence of an
# assignment. No DUF appears in the top 18, so panel A needs no second colour — the DUF share
# is stated in the title instead.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(14.5, 7.2),
                               gridspec_kw={"width_ratios": [1.45, 1]})

tf = (fn.groupby(["ID", "DE", "known_function"])["members"].sum()
      .sort_values(ascending=False).head(18).iloc[::-1].reset_index())
y = np.arange(len(tf))
axA.barh(y, tf["members"], height=0.44,
         color=[S1 if k else NEUT for k in tf["known_function"]])
for i, v in enumerate(tf["members"]):
    axA.text(v + tf["members"].max() * 0.012, i, f"{v:,}", va="center", fontsize=8, color=INK2)
axA.set(yticks=y, yticklabels=[f"{r.ID}  —  {r.DE[:40]}" for r in tf.itertuples()],
        xlim=(0, tf["members"].max() * 1.16))
axA.tick_params(axis="y", labelsize=8)
axA.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axA.set_xlabel("dark ORFs in families carrying this domain", fontsize=9, color=INK2)
duf_pct = fn.loc[~fn["known_function"], "members"].sum() / RESCUED * 100
axA.set_title(f"Top 18 Pfam families on the rescued dark proteome\n"
              f"{TOPF.sum() / RESCUED * 100:.0f}% of the {RESCUED:,} rescued ORFs · "
              f"{fn['acc'].nunique():,} Pfam families in all\n"
              f"none of the top 18 is a DUF — the {duf_pct:.1f}% DUF share sits in the tail "
              f"({fn.loc[~fn['known_function'], 'acc'].nunique():,} DUFs)",
              fontsize=10, color=INK, loc="left")
tidy(axA)

cl = CLAN.head(11).copy()
cl["(no clan assigned)"] = noclan
cl = cl.sort_values().iloc[-12:]
yb = np.arange(len(cl))
axB.barh(yb, cl.values, height=0.44,
         color=[NEUT if k.startswith("(no clan") else S1 for k in cl.index])
for i, v in enumerate(cl.values):
    axB.text(v + cl.max() * 0.015, i, f"{v / RESCUED * 100:.1f}%", va="center",
             fontsize=8, color=INK2)
axB.set(yticks=yb, yticklabels=cl.index, xlim=(0, cl.max() * 1.18))
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("dark ORFs", fontsize=9, color=INK2)
axB.set_title("Pfam clans — Pfam's own superfamily grouping\n"
              "replication, DNA binding and plasmid-toxin clans dominate\n"
              "grey = Pfam assigns no clan", fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


## 6. Does the dark plasmidome encode anti-defense systems?

A plausible hypothesis for what the dark proteome is *for*: **anti-defense**. Anti-CRISPR (Acr)
proteins are short (50–150 aa), hyperdiverse, and share no common fold — the exact profile of an
unannotatable ORF — and plasmids have every reason to carry them. Anti-SOS (`psiA`/`psiB`) and
anti-restriction (`ardA`, `klcA`, `ocr`) are classic conjugative-plasmid cargo.

Pfam is close to the wrong tool here: it holds ~11 Acr models against >100 experimentally
characterised Acr families. So this section searches **dbAPIS** (database of anti-Prokaryotic Immune
Systems) instead — **290 HMMs = 200 APIS families + 90 anti-CRISPR families**, roughly 8× Pfam's Acr
coverage.

Two complications, both handled explicitly rather than hidden:

1. **dbAPIS models carry no curated gathering thresholds**, unlike Pfam. A cutoff has to be chosen,
   so results are reported across three stringency tiers — the conclusion must not depend on which
   one you pick.
2. **Several dbAPIS models cross-hit abundant plasmid housekeeping proteins.** Every hit is
   therefore adjudicated against its Pfam assignment instead of being taken at face value.


In [ ]:
# R6 — does the dark plasmidome encode anti-defense systems (anti-CRISPR, anti-SOS,
# anti-restriction)? Searched against dbAPIS (anti-Prokaryotic Immune Systems), 290 HMMs =
# 200 APIS families + 90 anti-CRISPR families. Search: scripts/dbapis_search.sbatch.
#
# Two things make this harder than the Pfam run and are handled explicitly:
#   1. dbAPIS models carry NO curated GA thresholds, so a cutoff must be chosen -> report tiers.
#   2. Several dbAPIS models cross-hit abundant plasmid housekeeping proteins, so every hit is
#      adjudicated against its Pfam assignment rather than taken at face value.
DBR = ROOT / "data" / "dbapis_run"

rows = []
for line in open(DBR / "dbapis.domtbl"):
    if line.startswith("#"):
        continue
    p = line.split(None, 22)
    rows.append((p[0], p[3], int(p[5]), float(p[6]), int(p[15]), int(p[16])))
db = pd.DataFrame(rows, columns=["target", "model", "qlen", "evalue", "hmm_from", "hmm_to"])
db["set"] = db["target"].str.split("::").str[0]
db["hmm_cov"] = (db["hmm_to"] - db["hmm_from"] + 1) / db["qlen"]

apis = pd.read_csv(ROOT / "data/refs/dbapis/seed_and_familyrep_all_infor.tsv",
                   sep="\t", low_memory=False)
db["inhibits"] = db["model"].map(dict(zip(apis["APIS families"], apis["Defense systems"])))

print("stringency tiers (HMM coverage >= 0.5 throughout) — the answer must not depend on one:")
print(f"{'tier':<12}{'E<=':>8} | {'dark families':>14}{'dark ORFs':>11}{'% of dark':>11}")
for name, ev in [("permissive", 1e-3), ("standard", 1e-5), ("strict", 1e-10)]:
    s = db[(db.evalue <= ev) & (db.hmm_cov >= 0.5) & db.set.eq("DARKREP")].copy()
    s["rep"] = s["target"].str.split("::").str[1]
    s = s.sort_values("evalue").drop_duplicates("rep")
    n = s["rep"].map(size).sum()
    print(f"{name:<12}{ev:>8.0e} | {len(s):>14,}{n:>11,}{n / len(dark) * 100:>10.3f}%")

# Standard tier, then adjudicate each hit against Pfam.
dk = db[(db.evalue <= 1e-5) & (db.hmm_cov >= 0.5) & db.set.eq("DARKREP")].copy()
dk["rep"] = dk["target"].str.split("::").str[1]
dk = dk.sort_values("evalue").drop_duplicates("rep")
dk["members"] = dk["rep"].map(size)
dk = dk.merge(fn[["rep", "ID"]], on="rep", how="left").rename(columns={"ID": "pfamID"})

CONFOUND = re.compile(r"^(RepA_N|Bac_RepA_C|Rep_|Rep3|RepL|Replicase|SSB|MethyltransfD12|"
                      r"DNA_methylase|N6_M|Mob_|MobA|Relaxase|NikA|HTH_|Resolvase|"
                      r"Phage_integrase|MPTase)", re.I)
ANTI_OK = re.compile(r"^(ArdA|ArdcN|Antirestrict|Psi[AB]|ocr|Ral|Lar_|DM_Arn|Acr)", re.I)

def adjudicate(r):
    if pd.isna(r.pfamID):        return "no Pfam hit (unadjudicated)"
    if ANTI_OK.search(r.pfamID): return "Pfam agrees (anti-defense)"
    if CONFOUND.search(r.pfamID):return "Pfam contradicts (housekeeping)"
    return "Pfam names something else"
dk["verdict"] = dk.apply(adjudicate, axis=1)

print(f"\n=== adjudicating the {len(dk)} dark hits at the standard tier ===")
ADJ = (dk.groupby("verdict").agg(families=("rep", "size"), dark_ORFs=("members", "sum"))
       .sort_values("dark_ORFs", ascending=False))
ADJ["% of dark proteome"] = ADJ["dark_ORFs"] / len(dark) * 100
print(ADJ.round(4).to_string())

print("\nthe contradicted hits, and what Pfam actually calls them:")
print(dk[dk.verdict.str.startswith("Pfam contradicts")]
      .groupby(["model", "pfamID"])["members"].sum()
      .sort_values(ascending=False).head(8).to_string())

SURV = dk[~dk.verdict.str.startswith("Pfam contradicts")]
print("\n=== surviving anti-defense signal ===")
print(f"  {len(SURV)} dark families, {SURV['members'].sum():,} dark ORFs "
      f"= {SURV['members'].sum() / len(dark) * 100:.3f}% of the dark proteome")
# An Acr model IS the CRISPR-Cas case; anything else with no recorded system stays unlabelled.
sysname = np.where(SURV["model"].str.startswith("Acr"), "CRISPR-Cas (anti-CRISPR)",
                   SURV["inhibits"].fillna("(system not recorded)"))
BYSYS = SURV.groupby(sysname)["members"].sum().sort_values(ascending=False)
print("\n  by defense system inhibited:")
print(BYSYS.to_string())
acr = SURV[SURV["model"].str.startswith("Acr")]
print(f"\n  anti-CRISPR specifically: {len(acr)} families, {acr['members'].sum()} dark ORFs "
      f"({acr['members'].sum() / len(dark) * 100:.3f}% of the dark proteome)")
print(acr.groupby("model")["members"].sum().sort_values(ascending=False).to_string())


In [ ]:
# Figure R6. Panel A is a funnel, not a bar chart of the final number: the point is how much of
# the raw dbAPIS signal does not survive contact with Pfam. Grey = discarded / unverifiable.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5.2),
                               gridspec_kw={"width_ratios": [1.15, 1]})

order = ["Pfam contradicts (housekeeping)", "no Pfam hit (unadjudicated)",
         "Pfam names something else", "Pfam agrees (anti-defense)"]
cols = {"Pfam contradicts (housekeeping)": NEUT, "no Pfam hit (unadjudicated)": "#d4d3cc",
        "Pfam names something else": S4, "Pfam agrees (anti-defense)": S1}
vals = [ADJ.loc[k, "dark_ORFs"] for k in order]
left = 0.0
for k, v in zip(order, vals):
    axA.barh([0], [v], left=left, height=0.40, color=cols[k], edgecolor="white", linewidth=1.5,
             label=f"{k}  ({v})")
    left += v
axA.set(xlim=(0, sum(vals)), yticks=[], ylim=(-0.36, 0.36))
axA.set_xlabel(f"dark ORFs among the {len(dk)} dbAPIS hits (standard tier)",
               fontsize=9, color=INK2)
axA.legend(frameon=False, fontsize=8, labelcolor=INK2, loc="upper center",
           bbox_to_anchor=(0.5, -0.26), ncol=2)
axA.set_title(f"dbAPIS hits adjudicated against Pfam\n"
              f"{ADJ.loc['Pfam contradicts (housekeeping)', 'dark_ORFs'] / sum(vals) * 100:.0f}% "
              f"of the raw signal is cross-hits to plasmid housekeeping proteins",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

b = BYSYS.iloc[::-1]
yb = np.arange(len(b))
axB.barh(yb, b.values, height=0.42, color=S1)
for i, v in enumerate(b.values):
    axB.text(v + b.max() * 0.02, i, f"{v}", va="center", fontsize=9, color=INK2)
axB.set(yticks=yb, yticklabels=b.index, xlim=(0, b.max() * 1.2))
axB.set_xlabel("dark ORFs (surviving hits)", fontsize=9, color=INK2)
axB.set_title(f"What survives, by defense system inhibited\n"
              f"{SURV['members'].sum():,} dark ORFs total = "
              f"{SURV['members'].sum() / len(dark) * 100:.3f}% of the dark proteome",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

plt.tight_layout()
plt.show()


### Answer: no. Anti-defense does not explain the dark plasmidome

**158 dark ORFs — 0.042% of the dark proteome.** That is the surviving anti-defense signal after
adjudication, and it is three orders of magnitude short of explaining anything. The hypothesis is
cleanly rejected, at every stringency tier.

Two findings worth keeping:

**The raw dbAPIS number is inflated ~3×.** At face value the search reports 460 dark ORFs (0.122%).
**66% of that is cross-hits to abundant plasmid housekeeping proteins** — `APIS192`/`APIS107` hitting
DNA methyltransferases, `AcrIIA21` hitting `RepA_N` replication initiators (68 ORFs — the single
largest "anti-CRISPR" signal is a Rep protein), `APIS131` hitting `SSB`, `APIS108` hitting a
metallopeptidase. Taken uncritically this section would have reported ~3× the real number, and would
have called plasmid replication proteins anti-CRISPRs.

**Anti-SOS is not hiding in the dark — it is already annotated.** `psiA`/`psiB` appear 69 times in
the *named* proteome and only ~15 dark ORFs' worth in the dark one. Same for `ardA`/`ardC`/
`Antirestrict`. Where plasmids carry classic anti-defense genes, existing tools already find them;
they are not part of the dark fraction.

By system, of the 158 surviving: CRISPR-Cas 63 · restriction-modification 48 · unrecorded 24 ·
SOS response 15 · Retron 6 · NARP 1 · RecBCD 1.

### Limits of this negative result

- **Absence of evidence, bounded by database coverage.** dbAPIS holds 90 Acr families; the true
  number of Acr families is unknown and certainly larger. A novel Acr with no dbAPIS homolog is
  invisible here — which is exactly the population the dark proteome would contain if the
  hypothesis were true. **This result rules out *known* anti-defense families, not the hypothesis.**
- **The adjudication regex is ours**, like the R4 scope split. It is deliberately aggressive: it
  discards any hit whose Pfam call is plasmid housekeeping, which will also discard genuine
  anti-defense proteins that happen to carry an HTH or Rep-like domain. The 158 is a floor.
- **No genomic context was used.** Acr genes are typically flanked by an HTH-containing `aca` gene,
  and the PlasAnn shards carry CDS coordinates — a guilt-by-association scan is feasible and would
  be independent of homology. Not run here.


## Verdict: partly artifact, mostly not

The question was whether the dark plasmidome is a PlasAnn artifact or real. **It is measurably
both**, and the proportions are the result:

| slice of the dark proteome | % | reading |
|---|---:|---|
| no named homolog **and** no Pfam match | **57.4** | genuine dark matter |
| no named homolog, but Pfam names it | **28.5** | **PlasAnn annotation failure** |
| has a named homolog + Pfam match | 10.9 | inconsistent PlasAnn labelling |
| has a named homolog, no Pfam match | 3.2 | PlasAnn-specific knowledge |

**The artifact half is worse than a coverage gap.** Of what Pfam rescued, ~47% carries a domain
inside a category PlasAnn *already annotates* — `Rep_1`, `Rep3_N`, `Mob_Pre`, `NikA-like`,
`Relaxase`, `MobA_MobL`, `RepL`, `ParE_toxin`, `RelB`, `YoeB_toxin`, `TraD`, `T4SS-DNA_transf`.
That is **~18.5% of the entire dark proteome**: replication, mobilization and toxin–antitoxin
proteins that PlasAnn has categories for and still labelled "Open reading frame". Section 1's
finding that the dark proteome looks like replicate-and-mobilize machinery was right for a duller
reason than novelty — some of it literally *is* that machinery, unlabelled.

**The rest survives.** After Pfam, **3,343 substantial families (≥10 members) have no Pfam domain
at all**, holding **115,811 dark ORFs (30.6% of the dark proteome)**. **72.9% of them span ≥2 MOB
lineages**, median 3 lineages and 4 habitats. Those are in
`data/pfam_run/dark_novel_families.tsv` — the candidate list, now filtered by an external database
rather than by PlasAnn's silence.

### What the earlier sections must now be read with

Section 3's headline — "85.9% of dark ORFs have no named homolog" — still stands as computed, but
**it is no longer evidence of novelty**: a third of that 85.9% is recognisable to Pfam. The claim
that survives is narrower and better supported: *a large, lineage-spanning protein space in the
small plasmidome is invisible to both plasmid-specific annotation and to Pfam.*

### Remaining limits

- **Pfam is not the last word.** No hit at `--cut_ga` is not proof of novelty — structure-based
  search (foldseek vs AlphaFold DB) reaches further than profile HMMs, and is the next step.
- **Median Pfam alignment covers ~52% of the dark protein**, so even a rescued protein is often
  only partly explained.
- **Representative-based propagation**: members agree with their family representative's Pfam
  status **96.1%** of the time (200 families, 3,678 members), with 23/200 families internally
  mixed. Good, not perfect — family-level rates carry a few % of slop.
- **One clustering threshold** (30% id / 80% cov); untested at 40% and 50%.
- **`mob_cluster` is a mash-based proxy, not a phylogeny**; the subset is lineage-skewed
  (`AA379` holds 19,427 of 71,414).
- **PlasAnn's ORF calls are inherited, not verified** — no independent gene calling was run.
- Per-gene coverage is 92.2%; the 562-plasmid CARD/PlasAnn AMR definitional leak from
  `cryptic_plasmids.ipynb` cell 16 carries through unchanged (0.24%).

Two defects were found and fixed while running this, both documented in
`reports/pfam_dark_validation.md`: duplicate protein records from plasmids present in **both**
PlasAnn shard series (0.22%), and a CDS-indexing bug that silently broke the dark↔all-CDS join for
10.7% of dark ORFs. All numbers above are post-fix.
